In [1]:
!pip install kagglehub
!pip install tensorflow==2.13.0
# # !pip uninstall tensorflow -y


  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached kagglesdk-0.1.15-py3-none-any.whl.metadata (13 kB)
Using cached kagglehub-1.0.0-py3-none-any.whl (70 kB)
Using cached kagglesdk-0.1.15-py3-none-any.whl (160 kB)
  Using cached tensorflow-2.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.4 kB)
  Using cached gast-0.4.0-py3-none-any.whl.metadata (1.1 kB)
  Using cached keras-2.13.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached numpy-1.24.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
  Using cached protobuf-4.25.8-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
  Using cached tensorboard-2.13.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached tensorflow_estimator-2.13.0-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached typing_extensions-4.5.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached tensor

In [2]:
import pandas as pd
import numpy as np
import glob
import gc
import seaborn as sns
import matplotlib.pyplot as plt

# Models
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Preprocessing & Metrics
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os
import kagglehub

2026-02-13 11:18:09.653380: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-13 11:18:09.655464: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-13 11:18:09.702545: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-13 11:18:09.703664: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-13 11:18:10.602714: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

In [3]:
LABEL_MAP = {
    "BENIGN": 0,
    "Botnet": 1,
    "Botnet - Attempted": 2,
    "DDoS": 3,
    "DoS GoldenEye": 4,
    "DoS GoldenEye - Attempted": 5,
    "DoS Hulk": 6,
    "DoS Hulk - Attempted": 7,
    "DoS Slowhttptest": 8,
    "DoS Slowhttptest - Attempted": 9,
    "DoS Slowloris": 10,
    "DoS Slowloris - Attempted": 11,
    "FTP-Patator": 12,
    "FTP-Patator - Attempted": 13,
    "Heartbleed": 14,
    "Infiltration": 15,
    "Infiltration - Attempted": 16,
    "Infiltration - Portscan": 17,
    "Portscan": 18,
    "SSH-Patator": 19,
    "SSH-Patator - Attempted": 20,
    "Web Attack - Brute Force": 21,
    "Web Attack - Brute Force - Attempted": 22,
    "Web Attack - SQL Injection": 23,
    "Web Attack - SQL Injection - Attempted": 24,
    "Web Attack - XSS": 25,
    "Web Attack - XSS - Attempted": 26
}

# Invert map for reporting (0 -> "BENIGN")
REVERSE_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

IMPORTANT_COLS = [
    'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet',
    'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean',
    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Min', 'Fwd IAT Std',
    'Bwd IAT Std', 'Fwd PSH Flags', 'Bwd URG Flags', 'Fwd RST Flags',
    'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length',
    'Bwd Packets/s', 'Packet Length Min', 'FIN Flag Count',
    'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count',
    'URG Flag Count', 'CWR Flag Count', 'ECE Flag Count', 'Down/Up Ratio',
    'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg',
    'Bwd Bulk Rate Avg', 'Subflow Fwd Packets', 'Subflow Fwd Bytes',
    'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'FWD Init Win Bytes',
    'Bwd Init Win Bytes', 'Fwd Act Data Pkts', 'Fwd Seg Size Min',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'ICMP Code',
    'ICMP Type', 'Total TCP Flow Time', 
    'Timestamp', 
    'Label'
]

def load_and_process(file_list):
    """Loads files, filters cols, and applies label map."""
    print(f"Reading {len(file_list)} files...")
    # Generator for memory efficiency
    df = pd.concat((pd.read_csv(f) for f in file_list), ignore_index=True)
    
    # Filter Columns
    existing_cols = [c for c in IMPORTANT_COLS if c in df.columns]
    df = df[existing_cols]
    
    # Handle NaNs/Inf
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    
    # Sort by Timestamp for Time Series
    if 'Timestamp' in df.columns:
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
        df.sort_values('Timestamp', inplace=True)
        df.drop(columns=['Timestamp'], inplace=True)

    # Apply Label Map
    df['Label'] = df['Label'].map(LABEL_MAP)
    
    # Drop unknown labels
    if df['Label'].isna().any():
        print(f"Dropping {df['Label'].isna().sum()} rows with unknown labels.")
        df = df.dropna(subset=['Label'])
        
    df['Label'] = df['Label'].astype(int)
    return df

In [4]:
path = kagglehub.dataset_download("ernie55ernie/improved-cicids2017-and-csecicids2018")
search_path = os.path.join(path, "CICIDS2017_improved", "*.csv")
train_files = glob.glob(search_path)
train_files.append("Attack_Traffic_Dataset.csv")
print("--- 1. Loading Combined Training Data ---")
df_combined = load_and_process(train_files)

--- 1. Loading Combined Training Data ---
Reading 6 files...


In [5]:
X = df_combined.drop(columns=['Label'])
y = df_combined['Label']

In [6]:
print("Splitting Combined Data into Train/Val...")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Splitting Combined Data into Train/Val...


In [7]:
X_train.shape

(1749725, 58)

In [8]:
X_val.shape

(437432, 58)

In [9]:
del df_combined, X, y
gc.collect()

0

In [10]:
print("\n--- 2. Loading External Test Data (Attack_Traffic_Dataset1.csv) ---")
df_ext = load_and_process(["Attack_Traffic_Dataset1.csv"])
X_test_ext = df_ext.drop(columns=['Label'])
y_test_ext = df_ext['Label']
del df_ext
gc.collect()


--- 2. Loading External Test Data (Attack_Traffic_Dataset1.csv) ---
Reading 1 files...


0

In [11]:
print("\nScaling Data...")
scaler = StandardScaler()
# Fit ONLY on training split
X_train_scaled = scaler.fit_transform(X_train)
# Transform Validation and External Test
X_val_scaled = scaler.transform(X_val)
X_test_ext_scaled = scaler.transform(X_test_ext)


Scaling Data...


In [12]:
results = []

def evaluate_model(name, model, X_v, y_v, X_e, y_e):
    """Helper to predict and print scores for both sets"""
    print(f"Evaluating {name}...")
    
    # Internal Validation
    pred_val = model.predict(X_v)
    acc_val = accuracy_score(y_v, pred_val)
    
    # External Test
    pred_ext = model.predict(X_e)
    acc_ext = accuracy_score(y_e, pred_ext)
    
    print(f"  -> Val Acc: {acc_val:.4f} | Ext Test Acc: {acc_ext:.4f}")
    return {"Model": name, "Val_Acc": acc_val, "Ext_Test_Acc": acc_ext}

In [13]:
tf.config.list_physical_devices('GPU')
tf.__version__

2026-02-13 11:18:54.956589: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-13 11:18:54.960802: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


'2.13.0'

In [14]:
print("\n LSTM Time Series...")
# Reshape for LSTM
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_val_lstm = X_val_scaled.reshape((X_val_scaled.shape[0], 1, X_val_scaled.shape[1]))
X_ext_lstm = X_test_ext_scaled.reshape((X_test_ext_scaled.shape[0], 1, X_test_ext_scaled.shape[1]))

num_classes = len(LABEL_MAP)
y_train_ohe = to_categorical(y_train, num_classes=num_classes)
y_val_ohe = to_categorical(y_val, num_classes=num_classes)

model = Sequential()
model.add(LSTM(64, input_shape=(1, X_train_scaled.shape[1]), return_sequences=False))
model.add(Dropout(0.4))
model.add(Dense(num_classes, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(X_train_lstm, y_train_ohe, validation_data=(X_val_lstm, y_val_ohe), epochs=5, batch_size=1024, verbose=1)

# Predict LSTM
pred_probs_val = model.predict(X_val_lstm)
pred_probs_ext = model.predict(X_ext_lstm)
results.append({
    "Model": "LSTM", 
    "Val_Acc": accuracy_score(y_val, np.argmax(pred_probs_val, axis=1)),
    "Ext_Test_Acc": accuracy_score(y_test_ext, np.argmax(pred_probs_ext, axis=1))
})


 LSTM Time Series...
Epoch 1/5
1709/1709 [==============================] - 13s 7ms/step - loss: 0.2496 - accuracy: 0.9527 - val_loss: 0.0492 - val_accuracy: 0.9863
Epoch 2/5
1709/1709 [==============================] - 11s 6ms/step - loss: 0.0491 - accuracy: 0.9857 - val_loss: 0.0333 - val_accuracy: 0.9896
Epoch 3/5
1709/1709 [==============================] - 11s 6ms/step - loss: 0.0370 - accuracy: 0.9886 - val_loss: 0.0282 - val_accuracy: 0.9910
Epoch 4/5
1709/1709 [==============================] - 11s 6ms/step - loss: 0.0316 - accuracy: 0.9898 - val_loss: 0.0253 - val_accuracy: 0.9914
Epoch 5/5
2865/2865 [==============================] - 4s 1ms/step


In [15]:
print("\nCNN + LSTM Model...")

from tensorflow.keras.layers import Conv1D, MaxPooling1D

model_cnn_lstm = Sequential()

# CNN extracts local feature patterns
model_cnn_lstm.add(Conv1D(filters=64,
                          kernel_size=1,
                          activation='relu',
                          input_shape=(1, X_train_scaled.shape[1])))

model_cnn_lstm.add(MaxPooling1D(pool_size=1))

# LSTM captures sequential behavior
model_cnn_lstm.add(LSTM(64))

model_cnn_lstm.add(Dropout(0.4))
model_cnn_lstm.add(Dense(num_classes, activation='softmax'))

model_cnn_lstm.compile(loss='categorical_crossentropy',
                       optimizer='adam',
                       metrics=['accuracy'])

model_cnn_lstm.fit(X_train_lstm, y_train_ohe,
                   validation_data=(X_val_lstm, y_val_ohe),
                   epochs=5,
                   batch_size=256,
                   verbose=1)

# Predictions
pred_val = model_cnn_lstm.predict(X_val_lstm)
pred_ext = model_cnn_lstm.predict(X_ext_lstm)

results.append({
    "Model": "CNN + LSTM",
    "Val_Acc": accuracy_score(y_val, np.argmax(pred_val, axis=1)),
    "Ext_Test_Acc": accuracy_score(y_test_ext, np.argmax(pred_ext, axis=1))
})



CNN + LSTM Model...
Epoch 1/5
6835/6835 [==============================] - 32s 4ms/step - loss: 0.0708 - accuracy: 0.9822 - val_loss: 0.0280 - val_accuracy: 0.9894
Epoch 2/5
6835/6835 [==============================] - 29s 4ms/step - loss: 0.0261 - accuracy: 0.9915 - val_loss: 0.0215 - val_accuracy: 0.9913
Epoch 3/5
6835/6835 [==============================] - 29s 4ms/step - loss: 0.0214 - accuracy: 0.9927 - val_loss: 0.0186 - val_accuracy: 0.9934
Epoch 4/5
6835/6835 [==============================] - 29s 4ms/step - loss: 0.0196 - accuracy: 0.9932 - val_loss: 0.0175 - val_accuracy: 0.9938
Epoch 5/5
2865/2865 [==============================] - 4s 1ms/step


In [17]:
print("\n Transformer Model...")

from tensorflow.keras.layers import Input, MultiHeadAttention
from tensorflow.keras.layers import LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.layers import Add, Dense, Dropout
from tensorflow.keras.models import Model

inputs = Input(shape=(1, X_train_scaled.shape[1]))

# Multi-head self-attention
attention = MultiHeadAttention(num_heads=4, key_dim=32)(inputs, inputs)

# Residual connection + normalization
x = Add()([inputs, attention])
x = LayerNormalization()(x)

# Feed-forward network
ff = Dense(64, activation='relu')(x)
ff = Dense(X_train_scaled.shape[1])(ff)

# Residual connection + normalization
x = Add()([x, ff])
x = LayerNormalization()(x)

# Pooling
x = GlobalAveragePooling1D()(x)
x = Dropout(0.4)(x)

outputs = Dense(num_classes, activation='softmax')(x)

model_transformer = Model(inputs, outputs)

model_transformer.compile(loss='categorical_crossentropy',
                          optimizer='adam',
                          metrics=['accuracy'])

model_transformer.fit(X_train_lstm, y_train_ohe,
                      validation_data=(X_val_lstm, y_val_ohe),
                      epochs=5,
                      batch_size=256,
                      verbose=1)

# Predictions
pred_val = model_transformer.predict(X_val_lstm)
pred_ext = model_transformer.predict(X_ext_lstm)

results.append({
    "Model": "Transformer",
    "Val_Acc": accuracy_score(y_val, np.argmax(pred_val, axis=1)),
    "Ext_Test_Acc": accuracy_score(y_test_ext, np.argmax(pred_ext, axis=1))
})



 Transformer Model...
Epoch 1/5
6835/6835 [==============================] - 44s 6ms/step - loss: 0.0570 - accuracy: 0.9834 - val_loss: 0.0305 - val_accuracy: 0.9885
Epoch 2/5
6835/6835 [==============================] - 41s 6ms/step - loss: 0.0314 - accuracy: 0.9895 - val_loss: 0.0269 - val_accuracy: 0.9903
Epoch 3/5
6835/6835 [==============================] - 41s 6ms/step - loss: 0.0290 - accuracy: 0.9903 - val_loss: 0.0254 - val_accuracy: 0.9910
Epoch 4/5
6835/6835 [==============================] - 41s 6ms/step - loss: 0.0261 - accuracy: 0.9912 - val_loss: 0.0232 - val_accuracy: 0.9916
Epoch 5/5
2865/2865 [==============================] - 5s 2ms/step


In [18]:
print("\n" + "="*50)
print("FINAL EVALUATION SUMMARY")
print("="*50)
df_res = pd.DataFrame(results)
print(df_res.sort_values("Ext_Test_Acc", ascending=False))


FINAL EVALUATION SUMMARY
         Model   Val_Acc  Ext_Test_Acc
0         LSTM  0.992056      0.924999
2  Transformer  0.992303      0.921016
1   CNN + LSTM  0.993396      0.910084
